# BÀI THỰC HÀNH TUẦN 6 (Phần 2): ENSEMBLE LEARNING
## ÁP DỤNG HỌC ĐỒNG BỘ (BAGGING, BOOSTING, STACKING) TRÊN BÀI TOÁN PHÂN LOẠI

> **Dữ liệu:** Bank Marketing Dataset (UCI) — `bank-full.csv`  
> **Bài toán:** Phân loại nhị phân — Dự đoán khách hàng có đăng ký tiền gửi kỳ hạn hay không (`y`: yes/no)  
> **Tiếp nối:** Sử dụng cùng bộ dữ liệu và cách tiền xử lý với Phần 1 (Classification) để so sánh trực tiếp  

---

### Ensemble Learning là gì?

Thay vì dựa vào **1 mô hình đơn lẻ**, Ensemble Learning kết hợp **nhiều mô hình** lại để đưa ra dự đoán chính xác hơn. Có 3 chiến lược chính:

| Chiến lược | Ý tưởng | Ví dụ |
|-----------|---------|-------|
| **Bagging** | Huấn luyện song song nhiều mô hình trên các tập con ngẫu nhiên, rồi bỏ phiếu đa số | Random Forest |
| **Boosting** | Huấn luyện tuần tự, mô hình sau tập trung sửa lỗi của mô hình trước | Gradient Boosting, XGBoost, LightGBM |
| **Stacking** | Dùng dự đoán của nhiều mô hình cơ sở làm đầu vào cho mô hình meta | Stacking Classifier |

---

## 1. THIẾT LẬP MÔI TRƯỜNG & LOAD DỮ LIỆU

### 1.1. Import thư viện

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
np.seterr(all='ignore')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time

# Tiền xử lý
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score, learning_curve
)

# Mô hình cơ sở (base learners)
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier

# Ensemble methods
from sklearn.ensemble import (
    RandomForestClassifier,      # Bagging
    BaggingClassifier,           # Bagging tổng quát
    GradientBoostingClassifier,  # Boosting
    AdaBoostClassifier,          # Boosting
    StackingClassifier,          # Stacking
    VotingClassifier,            # Voting
)

# Đánh giá
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    roc_curve, ConfusionMatrixDisplay
)

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams.update({'font.size': 11, 'figure.dpi': 100})
np.random.seed(42)

print("Đã nạp xong thư viện.")

### 1.2. Load & tiền xử lý dữ liệu (giống Phần 1)

In [ ]:
df = pd.read_csv('../data/bank-full.csv', sep=';')
print(f"Kích thước dữ liệu: {df.shape[0]:,} dòng x {df.shape[1]} cột")

# Mã hóa (giống hệt Phần 1)
df_processed = df.copy()
df_processed['y'] = (df_processed['y'] == 'yes').astype(int)
for col in ['default', 'housing', 'loan']:
    df_processed[col] = (df_processed[col] == 'yes').astype(int)
cat_cols = ['job', 'marital', 'education', 'contact', 'month', 'poutcome']
df_processed = pd.get_dummies(df_processed, columns=cat_cols, drop_first=True)

# Chia dữ liệu (giống hệt Phần 1)
X = df_processed.drop('y', axis=1)
y = df_processed['y']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Chuẩn hóa
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train: {X_train.shape[0]:,} mẫu | Test: {X_test.shape[0]:,} mẫu")
print(f"Tỷ lệ lớp 'yes': {y_train.mean():.2%}")
print(f"Số đặc trưng: {X_train.shape[1]}")

---
## 2. CHIẾN LƯỢC 1: BAGGING (Bootstrap Aggregating)

### Nguyên lý hoạt động:
1. Tạo N tập con bằng cách **lấy mẫu ngẫu nhiên có hoàn lại** (bootstrap) từ tập train
2. Huấn luyện N mô hình cùng loại song song trên N tập con
3. Kết hợp kết quả bằng **bỏ phiếu đa số** (classification) hoặc trung bình (regression)

→ Giảm **variance** (sự biến thiên), giúp các mô hình không ổn định (như Decision Tree) trở nên ổn định hơn.

### 2.1. Random Forest — Hiện thực phổ biến nhất của Bagging

In [ ]:
# Random Forest không cần chuẩn hóa
rf_model = RandomForestClassifier(
    n_estimators=300,         # Số cây trong rừng
    max_depth=12,             # Giới hạn độ sâu mỗi cây
    min_samples_split=30,     # Ít nhất 30 mẫu để chia nút
    min_samples_leaf=10,      # Ít nhất 10 mẫu mỗi lá
    max_features='sqrt',      # Mỗi lần chia, chỉ xét √p đặc trưng ngẫu nhiên
    class_weight='balanced',  # Xử lý mất cân bằng lớp
    n_jobs=-1,                # Dùng tất cả CPU cores
    random_state=42
)

start = time.time()
rf_model.fit(X_train, y_train)
rf_time = time.time() - start

y_pred_rf = rf_model.predict(X_test)
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]

print(f"Thời gian huấn luyện: {rf_time:.2f}s")
print(f"Số cây: {rf_model.n_estimators}")
print(f"\n{classification_report(y_test, y_pred_rf, target_names=['No', 'Yes'])}")

### 2.2. Feature Importance từ Random Forest

In [ ]:
feat_imp = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
feat_imp.head(15).plot(kind='barh', ax=ax, color='forestgreen', edgecolor='white')
ax.set_xlabel('Feature Importance (Mean Decrease Impurity)')
ax.set_title('Top 15 đặc trưng quan trọng — Random Forest (300 cây)', fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print("\nSo sánh với Decision Tree đơn lẻ (Phần 1):")
print("→ Random Forest cho Feature Importance ổn định hơn vì trung bình từ 300 cây.")
print("→ Giảm thiên kiến (bias) của việc chỉ dựa vào 1 cây duy nhất.")

### 2.3. Hiệu ứng Bagging: Số cây vs Hiệu suất

In [ ]:
# Theo dõi hiệu suất khi tăng số cây
n_tree_range = [1, 5, 10, 25, 50, 100, 150, 200, 300]
oob_scores = []
test_aucs = []

for n in n_tree_range:
    rf_temp = RandomForestClassifier(
        n_estimators=n, max_depth=12, min_samples_split=30,
        min_samples_leaf=10, max_features='sqrt',
        class_weight='balanced', oob_score=True,
        n_jobs=-1, random_state=42
    )
    rf_temp.fit(X_train, y_train)
    oob_scores.append(rf_temp.oob_score_)
    y_prob_temp = rf_temp.predict_proba(X_test)[:, 1]
    test_aucs.append(roc_auc_score(y_test, y_prob_temp))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(n_tree_range, oob_scores, 'o-', color='steelblue', linewidth=2, label='OOB Accuracy')
ax.plot(n_tree_range, test_aucs, 's--', color='coral', linewidth=2, label='Test AUC-ROC')
ax.set_xlabel('Số cây (n_estimators)')
ax.set_ylabel('Score')
ax.set_title('Hiệu ứng Bagging: Thêm cây → Hiệu suất ổn định hơn', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("→ Quan sát: Khi số cây tăng, hiệu suất ổn định và không overfitting.")
print("  Đây là ưu điểm cốt lõi của Bagging so với Decision Tree đơn lẻ.")

---
## 3. CHIẾN LƯỢC 2: BOOSTING (Tăng cường tuần tự)

### Nguyên lý hoạt động:
1. Huấn luyện mô hình đầu tiên trên toàn bộ dữ liệu
2. Xác định các mẫu bị **dự đoán sai** → Tăng trọng số cho chúng
3. Huấn luyện mô hình tiếp theo **tập trung vào các mẫu khó**
4. Lặp lại, kết hợp tất cả bằng trọng số

→ Giảm **bias** (sai lệch), biến các mô hình yếu (weak learners) thành mô hình mạnh.

### 3.1. Gradient Boosting

In [ ]:
gb_model = GradientBoostingClassifier(
    n_estimators=200,         # Số bước boosting
    learning_rate=0.1,        # Tốc độ học — nhỏ hơn → cần nhiều cây hơn nhưng tốt hơn
    max_depth=4,              # Cây nông (weak learner) — khác với Random Forest
    min_samples_split=50,
    min_samples_leaf=20,
    subsample=0.8,            # Lấy 80% mẫu mỗi bước → Stochastic Gradient Boosting
    random_state=42
)

start = time.time()
gb_model.fit(X_train, y_train)
gb_time = time.time() - start

y_pred_gb = gb_model.predict(X_test)
y_prob_gb = gb_model.predict_proba(X_test)[:, 1]

print(f"Thời gian huấn luyện: {gb_time:.2f}s")
print(f"\n{classification_report(y_test, y_pred_gb, target_names=['No', 'Yes'])}")

### 3.2. AdaBoost (Adaptive Boosting)

In [ ]:
ada_model = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=2),  # Cây rất nông (stump-like)
    n_estimators=200,
    learning_rate=0.1,
    algorithm='SAMME.R',    # SAMME.R dùng xác suất, nhanh và tốt hơn SAMME
    random_state=42
)

start = time.time()
ada_model.fit(X_train, y_train)
ada_time = time.time() - start

y_pred_ada = ada_model.predict(X_test)
y_prob_ada = ada_model.predict_proba(X_test)[:, 1]

print(f"Thời gian huấn luyện: {ada_time:.2f}s")
print(f"\n{classification_report(y_test, y_pred_ada, target_names=['No', 'Yes'])}")

### 3.3. Quá trình Boosting: Stage-wise Performance

In [ ]:
# Theo dõi hiệu suất qua từng bước boosting của Gradient Boosting
from sklearn.metrics import log_loss

# staged_predict_proba cho phép theo dõi từng bước
train_scores = []
test_scores = []
for i, (y_prob_train_stage, y_prob_test_stage) in enumerate(zip(
    gb_model.staged_predict_proba(X_train),
    gb_model.staged_predict_proba(X_test)
)):
    train_scores.append(roc_auc_score(y_train, y_prob_train_stage[:, 1]))
    test_scores.append(roc_auc_score(y_test, y_prob_test_stage[:, 1]))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(1, len(train_scores)+1), train_scores, label='Train AUC', color='steelblue', linewidth=2)
ax.plot(range(1, len(test_scores)+1), test_scores, label='Test AUC', color='coral', linewidth=2)
ax.set_xlabel('Số bước Boosting (n_estimators)')
ax.set_ylabel('AUC-ROC')
ax.set_title('Gradient Boosting — Hiệu suất theo từng bước boosting', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Đánh dấu điểm tốt nhất trên test
best_n = np.argmax(test_scores) + 1
best_auc = max(test_scores)
ax.axvline(x=best_n, color='gray', linestyle='--', alpha=0.5)
ax.annotate(f'Best: n={best_n}, AUC={best_auc:.4f}',
            xy=(best_n, best_auc), xytext=(best_n+20, best_auc-0.01),
            arrowprops=dict(arrowstyle='->', color='gray'), fontsize=10)

plt.tight_layout()
plt.show()

print(f"→ Test AUC tốt nhất tại n_estimators = {best_n}: AUC = {best_auc:.4f}")
print(f"  Sau đó nếu tiếp tục boosting, gap train-test tăng → Overfitting")

---
## 4. CHIẾN LƯỢC 3: VOTING & STACKING

### 4.1. Voting Classifier (Bỏ phiếu)

Kết hợp các mô hình **khác nhau về bản chất** (không phải cùng loại như Bagging/Boosting) bằng cách bỏ phiếu.

In [ ]:
# Voting Classifier: kết hợp các mô hình khác nhau
voting_model = VotingClassifier(
    estimators=[
        ('rf', RandomForestClassifier(n_estimators=100, max_depth=10,
                                      class_weight='balanced', random_state=42)),
        ('gb', GradientBoostingClassifier(n_estimators=100, max_depth=4,
                                          learning_rate=0.1, random_state=42)),
        ('lr', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)),
    ],
    voting='soft',  # 'soft' dùng xác suất trung bình (tốt hơn 'hard' bỏ phiếu đa số)
    n_jobs=-1
)

start = time.time()
voting_model.fit(X_train_scaled, y_train)
voting_time = time.time() - start

y_pred_voting = voting_model.predict(X_test_scaled)
y_prob_voting = voting_model.predict_proba(X_test_scaled)[:, 1]

print(f"Thời gian huấn luyện: {voting_time:.2f}s")
print(f"Các mô hình cơ sở: RF, Gradient Boosting, Logistic Regression")
print(f"\n{classification_report(y_test, y_pred_voting, target_names=['No', 'Yes'])}")

### 4.2. Stacking Classifier (Xếp chồng)

**Ý tưởng:** Dùng dự đoán của các mô hình cơ sở (level-0) làm **đặc trưng đầu vào** cho một mô hình meta (level-1).

```
Level 0:  Random Forest ──┐
          Gradient Boosting ──┤──→ [predictions] ──→ Level 1: Logistic Regression ──→ Final
          Naive Bayes ────┘
```

In [ ]:
stacking_model = StackingClassifier(
    estimators=[
        ('rf', RandomForestClassifier(n_estimators=100, max_depth=10,
                                      class_weight='balanced', random_state=42)),
        ('gb', GradientBoostingClassifier(n_estimators=100, max_depth=4,
                                          learning_rate=0.1, random_state=42)),
        ('nb', GaussianNB()),
    ],
    final_estimator=LogisticRegression(max_iter=1000, random_state=42),  # Meta-learner
    cv=5,           # 5-fold CV để tạo predictions cho level-1
    stack_method='predict_proba',  # Dùng xác suất thay vì nhãn
    n_jobs=-1
)

start = time.time()
stacking_model.fit(X_train_scaled, y_train)
stacking_time = time.time() - start

y_pred_stacking = stacking_model.predict(X_test_scaled)
y_prob_stacking = stacking_model.predict_proba(X_test_scaled)[:, 1]

print(f"Thời gian huấn luyện: {stacking_time:.2f}s")
print(f"Level-0: RF, Gradient Boosting, Naive Bayes")
print(f"Level-1 (Meta): Logistic Regression")
print(f"\n{classification_report(y_test, y_pred_stacking, target_names=['No', 'Yes'])}")

---
## 5. SO SÁNH TỔNG HỢP TẤT CẢ CÁC MÔ HÌNH ENSEMBLE

### 5.1. Bảng so sánh

In [ ]:
all_models = {
    'Random Forest (Bagging)':   (y_pred_rf, y_prob_rf, rf_time),
    'Gradient Boosting':         (y_pred_gb, y_prob_gb, gb_time),
    'AdaBoost':                  (y_pred_ada, y_prob_ada, ada_time),
    'Voting (RF+GB+LR)':        (y_pred_voting, y_prob_voting, voting_time),
    'Stacking (RF+GB+NB→LR)':   (y_pred_stacking, y_prob_stacking, stacking_time),
}

results = []
for name, (y_pred, y_prob, train_time) in all_models.items():
    results.append({
        'Mô hình': name,
        'Chiến lược': 'Bagging' if 'Bagging' in name else ('Boosting' if name in ['Gradient Boosting', 'AdaBoost'] else 'Combination'),
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision (Yes)': precision_score(y_test, y_pred),
        'Recall (Yes)': recall_score(y_test, y_pred),
        'F1 (Yes)': f1_score(y_test, y_pred),
        'AUC-ROC': roc_auc_score(y_test, y_prob),
        'Thời gian (s)': train_time
    })

df_results = pd.DataFrame(results).set_index('Mô hình')

def highlight_max(s):
    if s.name in ['Thời gian (s)', 'Chiến lược']:
        if s.name == 'Thời gian (s)':
            return ['font-weight: bold' if v == s.min() else '' for v in s]
        return ['' for _ in s]
    return ['font-weight: bold' if v == s.max() else '' for v in s]

display(df_results.style.apply(highlight_max).format({
    'Accuracy': '{:.4f}', 'Precision (Yes)': '{:.4f}',
    'Recall (Yes)': '{:.4f}', 'F1 (Yes)': '{:.4f}',
    'AUC-ROC': '{:.4f}', 'Thời gian (s)': '{:.2f}'
}))

### 5.2. Confusion Matrix so sánh

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(25, 4))

for ax, (name, (y_pred, y_prob, _)) in zip(axes, all_models.items()):
    cm = confusion_matrix(y_test, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=['No', 'Yes']).plot(ax=ax, cmap='Greens', colorbar=False)
    short_name = name.split('(')[0].strip() if '(' in name else name
    ax.set_title(short_name, fontweight='bold', fontsize=10)

plt.suptitle('Confusion Matrix — So sánh các mô hình Ensemble', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 5.3. ROC Curve so sánh

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#4CAF50', '#2196F3', '#FF9800', '#9C27B0', '#E91E63']

for (name, (_, y_prob, _)), color in zip(all_models.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', color=color, linewidth=2)

ax.plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Random (AUC=0.500)')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curve — So sánh Ensemble Methods', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 6. SO SÁNH VỚI MÔ HÌNH ĐƠN LẺ (Phần 1)

### Ensemble có thực sự tốt hơn mô hình đơn?

In [ ]:
# Huấn luyện lại các mô hình đơn để so sánh trực tiếp
from sklearn.neural_network import MLPClassifier

single_models = {
    'Decision Tree (đơn)': DecisionTreeClassifier(
        max_depth=8, min_samples_split=50, min_samples_leaf=20,
        class_weight='balanced', random_state=42
    ),
    'Naive Bayes (đơn)': GaussianNB(),
    'Neural Network (đơn)': MLPClassifier(
        hidden_layer_sizes=(128, 64, 32), activation='relu',
        max_iter=300, early_stopping=True, random_state=42
    ),
}

comparison = []

for name, model in single_models.items():
    if 'Neural' in name or 'Naive' in name:
        model.fit(X_train_scaled, y_train)
        y_prob_temp = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, y_train)
        y_prob_temp = model.predict_proba(X_test)[:, 1]
    y_pred_temp = model.predict(X_test_scaled if ('Neural' in name or 'Naive' in name) else X_test)
    comparison.append({
        'Mô hình': name,
        'Loại': '🔹 Đơn lẻ',
        'F1 (Yes)': f1_score(y_test, y_pred_temp),
        'AUC-ROC': roc_auc_score(y_test, y_prob_temp),
    })

# Thêm các ensemble
for name, (y_pred, y_prob, _) in all_models.items():
    comparison.append({
        'Mô hình': name,
        'Loại': '🔸 Ensemble',
        'F1 (Yes)': f1_score(y_test, y_pred),
        'AUC-ROC': roc_auc_score(y_test, y_prob),
    })

df_compare = pd.DataFrame(comparison).set_index('Mô hình')
df_compare = df_compare.sort_values('AUC-ROC', ascending=False)
display(df_compare.style.format({'F1 (Yes)': '{:.4f}', 'AUC-ROC': '{:.4f}'}))

print("\n→ Nhận xét: Các phương pháp Ensemble thường vượt trội hơn mô hình đơn lẻ.")
print("  Boosting thường cho AUC-ROC cao nhất, Bagging cho kết quả ổn định nhất.")

### So sánh bằng biểu đồ

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

df_plot = df_compare.reset_index()
colors_map = {'🔹 Đơn lẻ': '#90CAF9', '🔸 Ensemble': '#FF8A65'}
bar_colors = [colors_map[t] for t in df_plot['Loại']]

for ax, metric in zip(axes, ['F1 (Yes)', 'AUC-ROC']):
    bars = ax.barh(df_plot['Mô hình'], df_plot[metric], color=bar_colors, edgecolor='white')
    for bar, val in zip(bars, df_plot[metric]):
        ax.text(val + 0.002, bar.get_y() + bar.get_height()/2, f'{val:.4f}',
                va='center', fontweight='bold', fontsize=9)
    ax.set_xlabel(metric)
    ax.set_title(f'{metric} — Đơn lẻ vs Ensemble', fontweight='bold')
    ax.invert_yaxis()

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#90CAF9', label='Mô hình Đơn lẻ'),
                   Patch(facecolor='#FF8A65', label='Mô hình Ensemble')]
fig.legend(handles=legend_elements, loc='lower center', ncol=2, fontsize=11,
           bbox_to_anchor=(0.5, -0.02))

plt.suptitle('So sánh Mô hình Đơn lẻ vs Ensemble — Bank Marketing', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 7. KẾT LUẬN & ĐÁNH GIÁ TỔNG QUAN

In [ ]:
print("="*80)
print("TỔNG KẾT — ENSEMBLE LEARNING TRÊN BANK MARKETING DATASET")
print("="*80)

best_ensemble_f1 = df_results['F1 (Yes)'].idxmax()
best_ensemble_auc = df_results['AUC-ROC'].idxmax()

print(f"\n📊 Mô hình Ensemble tốt nhất theo:")
print(f"   • F1-Score (Yes): {best_ensemble_f1} ({df_results.loc[best_ensemble_f1, 'F1 (Yes)']:.4f})")
print(f"   • AUC-ROC:        {best_ensemble_auc} ({df_results.loc[best_ensemble_auc, 'AUC-ROC']:.4f})")

print(f"\n" + "="*80)
print("SO SÁNH 3 CHIẾN LƯỢC ENSEMBLE")
print("="*80)

strategies = [
    ("1. BAGGING (Random Forest)", [
        "✅ Giảm variance → Ổn định, ít overfitting",
        "✅ Huấn luyện song song → Nhanh",
        "✅ Cung cấp Feature Importance ổn định",
        "✅ Không cần chuẩn hóa dữ liệu",
        "⚠️ Hiệu suất có thể không cao bằng Boosting trên dữ liệu mất cân bằng",
    ]),
    ("2. BOOSTING (Gradient Boosting, AdaBoost)", [
        "✅ Giảm bias → Hiệu suất thường cao nhất",
        "✅ Tập trung vào các mẫu khó → Tốt cho lớp thiểu số (yes)",
        "⚠️ Huấn luyện tuần tự → Chậm hơn Bagging",
        "⚠️ Dễ overfitting nếu boosting quá nhiều bước hoặc learning_rate quá lớn",
        "⚠️ Nhạy cảm với outliers (đặc biệt AdaBoost)",
    ]),
    ("3. STACKING & VOTING", [
        "✅ Kết hợp điểm mạnh của nhiều mô hình khác nhau",
        "✅ Thường cho kết quả ổn định nhất",
        "⚠️ Phức tạp nhất — khó điều chỉnh và debug",
        "⚠️ Thời gian huấn luyện = tổng thời gian tất cả mô hình cơ sở",
    ]),
]

for title, points in strategies:
    print(f"\n{title}")
    for p in points:
        print(f"   {p}")

print(f"\n" + "="*80)
print("KHUYẾN NGHỊ CUỐI CÙNG")
print("="*80)
print("""
• Nếu cần MÔ HÌNH TỐT NHẤT cho production:    → Gradient Boosting (hoặc XGBoost/LightGBM)
• Nếu cần ỔN ĐỊNH & DỄ DÙNG:                  → Random Forest
• Nếu cần KẾT HỢP nhiều góc nhìn:              → Stacking hoặc Voting
• Nếu cần DỄ DIỄN GIẢI cho doanh nghiệp:       → Decision Tree đơn (Phần 1) hoặc RF + SHAP

→ Ensemble Learning gần như luôn cải thiện hiệu suất so với mô hình đơn lẻ,
  đánh đổi bằng thời gian huấn luyện và độ phức tạp tăng lên.
""")